# 实验笔记：Ingest 缓存机制 Bug 修复

**日期**：2026-03-08  
**涉及文件**：`src/storage/ingest_manager.py`、`src/rag/pipeline.py`

---

## 问题背景

项目的 RAG pipeline 在启动时会调用 `ingest()` 将 `data/` 目录下的 PDF 文件处理并写入 Milvus 向量数据库。  
为了避免每次重启都重新解析（LlamaParse API 调用昂贵），新加入了 `IngestManager` 做缓存。

---

## 发现的 Bug

**现象**：用户替换或新增了 PDF 文件后，重启服务，RAG 检索结果仍然基于旧内容，新文件完全没有生效。

**根本原因（3 处）**：

### Bug 1 — `pipeline.py` 重复实现了指纹检查逻辑，并直接访问私有属性

`IngestManager` 已提供 `check_ingest_status()` 方法，但 `pipeline.py` 没有调用它，而是自己手写了一遍指纹比对，并直接操作 `manager._manifest`、`manager._file_hash()`、`manager._save_manifest()` 等私有属性：

```python
# 旧代码（pipeline.py）— 绕过了封装
cached_manifest = manager._manifest.get(col_name, {})
cached_files = cached_manifest.get("files", {})
for filepath in pdf_files:
    current_hash = manager._file_hash(filepath)  # 访问私有方法
    if cached_files.get(filename) != current_hash:
        changed_files.append(filepath)
# ...
manager._manifest[col_name]["last_ingested"] = manager._get_timestamp()  # 直接写私有属性
manager._save_manifest()  # 直接调用私有方法
```

结果 `IngestManager.check_ingest_status()` **完全是死代码**，从未被调用过。

### Bug 2 — `mark_ingested()` 记录范围错误，导致新文件被漏掉

旧的 `mark_ingested(data_dir, col_name)` 会扫描整个 `data_dir` 目录，把**所有 PDF 的 hash 都写入 manifest**，包括那些实际上根本没被处理的文件（因为 `file_filter` 限制）。

下次启动时，manifest 显示这些文件「已处理」，触发 skip，但 Milvus 里根本没有这些文件的内容。

```python
# 旧代码 — 记录了所有文件，但实际只 ingest 了 file_filter 匹配的那一个
def mark_ingested(self, data_dir: str, col_name: str):
    for f in os.listdir(data_dir):  # 扫描整个目录
        if f.lower().endswith(".pdf"):
            files[f] = self._file_hash(filepath)  # 全部记录进去
```

### Bug 3 — `check_ingest_status()` 在没有 PDF 文件时错误返回 `skip=True`

```python
# 旧代码 — 没有文件居然返回「可以跳过」
if not current_files:
    return IngestStatus(
        skip=True,   # ← 语义错误：无文件 ≠ 可以跳过
        reason=f"No PDF files found in {data_dir}",
    )
```

---

## 修复方案

| 位置 | 修改内容 |
|------|---------|
| `ingest_manager.py` | `check_ingest_status(data_dir, ...)` → 接收 `pdf_files: list[str]`（已过滤好的列表），不再自己扫目录 |
| `ingest_manager.py` | `mark_ingested(data_dir, ...)` → `mark_ingested_files(col_name, file_paths)`，只记录实际 ingest 的文件 |
| `ingest_manager.py` | 修复：无文件时返回 `skip=False` |
| `pipeline.py` | 删除重复的指纹检查逻辑，改为调用 `manager.check_ingest_status(...)` |
| `pipeline.py` | 删除私有属性访问，改为调用 `manager.mark_ingested_files(...)` |
| `pipeline.py` | `import os` 移到文件顶部 |


---

## 原始测试脚本（保留作参照）

以下是最初用于验证 Milvus hybrid search 是否正常工作的脚本。

In [ ]:
import sys
sys.path.insert(0, "..")

In [ ]:
from storage.vector_store import MilvusVectorStore
from rag.embedder import embed_texts, load_bge_m3_embedder, embed_query
from rag.chunker import chunk_text
from rag.parser import build_llama_parser, parse_documents, merge_documents
from rag.retriever import dense_search, sparse_search, hybrid_search

In [ ]:
# 解析 PDF、分块、向量化、写入 Milvus
embedder = load_bge_m3_embedder()

parser = build_llama_parser()
documents = parse_documents("../../data", parser)
all_doc = merge_documents(documents)
chunks = chunk_text(all_doc, max_chunk_size=300, hard_max_length=512)
chunks_embeddings = embed_texts(chunks, embedder)

store = MilvusVectorStore(col_name="test_collection", dense_dim=1024)
store.insert(chunks, chunks_embeddings)

print(f"插入 {len(chunks)} 个 chunks")

In [ ]:
# 检索对比：dense / sparse / hybrid
query = '什么是以人为本的座舱'
query_embedding = embed_query(query, embedder)

query_dense = query_embedding["dense"][0]
query_sparse = query_embedding["sparse"]._getrow(0)

dense_results = dense_search(store.col, query_dense)
sparse_results = sparse_search(store.col, query_sparse)
hybrid_results = hybrid_search(store.col, query_dense, query_sparse,
                               sparse_weight=0.7, dense_weight=1.0)

print("=== Dense Top-3 ===")
print(dense_results[:3])

print("\n=== Sparse Top-3 ===")
print(sparse_results[:3])

print("\n=== Hybrid Top-3 ===")
print(hybrid_results[:3])

---

## 修复后验证：`IngestManager` 缓存行为

In [ ]:
from storage.ingest_manager import IngestManager
import os

manager = IngestManager()
DATA_DIR = "../../data"
COL_NAME = "nio_ec6"

# 收集 PDF 文件（模拟 pipeline.py 的行为）
pdf_files = [
    os.path.join(DATA_DIR, f)
    for f in os.listdir(DATA_DIR)
    if f.lower().endswith(".pdf") and f == "EC6.pdf"
]
print(f"找到文件：{pdf_files}")

In [ ]:
# 场景 1：首次运行（manifest 为空）
status = manager.check_ingest_status(
    pdf_files=pdf_files,
    col_name=COL_NAME,
    collection_exists=False,
    collection_count=0,
)
print(f"skip={status.skip}")
print(f"reason={status.reason}")
print(f"changed_files={status.changed_files}")

In [ ]:
# 模拟 ingest 完成，记录 manifest
manager.mark_ingested_files(COL_NAME, pdf_files)
print(f"manifest 已写入：{manager._manifest.get(COL_NAME)}")

In [ ]:
# 场景 2：再次运行，文件没有变化 → 应该 skip
status = manager.check_ingest_status(
    pdf_files=pdf_files,
    col_name=COL_NAME,
    collection_exists=True,
    collection_count=500,
)
print(f"skip={status.skip}")
print(f"reason={status.reason}")
# 期望：skip=True

In [ ]:
# 场景 3：模拟文件被修改（手动破坏 manifest 里的 hash）
manager._manifest[COL_NAME]["files"]["EC6.pdf"] = "fake_hash_000"

status = manager.check_ingest_status(
    pdf_files=pdf_files,
    col_name=COL_NAME,
    collection_exists=True,
    collection_count=500,
)
print(f"skip={status.skip}")
print(f"reason={status.reason}")
print(f"changed_files={[os.path.basename(f) for f in status.changed_files]}")
# 期望：skip=False，能检测到文件变化

---

## 结论

- **修复前**：`pipeline.py` 绕过 `IngestManager` 的封装，自行管理 manifest，逻辑分散且不一致；`mark_ingested()` 记录范围过宽，导致新文件被误判为「已处理」而跳过。
- **修复后**：`pipeline.py` 完全通过 `IngestManager` 的公开接口操作（`check_ingest_status` + `mark_ingested_files`），职责清晰；只记录实际 ingest 的文件，文件变化能被正确检测到。